# ChurnShield model development

This notebook is the model-development record for the IBM telco churn CSV. It uses the same cleaning, feature, and comparison code as `backend.ml.train`.

The held-out test set is created before model comparison. It is scored only after the feature set, model, imbalance setting, and decision threshold have been chosen from the development data.

Figures and sentences in later cells are computed from this file. They are not copied from a paper.

In [ ]:

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.ml.experiment import (
    build_estimator,
    build_preprocessor,
    encode_target,
    load_clean_frame,
    run_experiments,
    split_development_test,
)
from backend.ml.explain import explain_one
from backend.ml.features import OPTIONAL_SERVICE_COLUMNS, add_domain_features
from backend.ml.preprocessing import FEATURE_COLUMNS, ID_COL, TARGET_COL, clean_features

DATA = ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

C:\Users\Ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset loading and inspection

The file is loaded as text first so that blank `TotalCharges` values are still visible. Pandas would otherwise decide the column type on its own.

In [2]:
raw = pd.read_csv(DATA, dtype=str)
print("shape", raw.shape)
print("columns", list(raw.columns))
display(raw.head())
display(raw.dtypes.astype(str).rename("dtype").to_frame())

shape (7043, 21)
columns ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


,dtype
customerID,str
gender,str
SeniorCitizen,str
Partner,str
Dependents,str
tenure,str
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


In [3]:
numeric_as_numbers = raw.copy()
for column in ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]:
    numeric_as_numbers[column] = pd.to_numeric(numeric_as_numbers[column], errors="coerce")

categorical = [column for column in raw.columns if column not in {ID_COL, TARGET_COL, "tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"}]
print("categorical columns", categorical)
print("duplicate rows", int(raw.duplicated().sum()))
print("duplicate customer IDs", int(raw[ID_COL].duplicated().sum()))
print("target counts")
display(raw[TARGET_COL].value_counts(dropna=False).to_frame("count"))
display((raw[TARGET_COL].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

categorical columns ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
duplicate rows 0
duplicate customer IDs 0
target counts


,count
Churn,
No,5174
Yes,1869


,percent
Churn,
No,73.46
Yes,26.54


In [4]:
missing_counts = raw.replace(r"^\s*$", np.nan, regex=True).isna().sum()
missing = missing_counts.rename("missing_count").to_frame()
missing["missing_percent"] = (missing["missing_count"] / len(raw) * 100).round(3)
display(missing)
print("descriptive statistics for fields that can be read as numbers")
display(numeric_as_numbers[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]].describe().round(3))

,missing_count,missing_percent
customerID,0,0.000
gender,0,0.000
SeniorCitizen,0,0.000
Partner,0,0.000
Dependents,0,0.000
tenure,0,0.000
PhoneService,0,0.000
MultipleLines,0,0.000
InternetService,0,0.000
OnlineSecurity,0,0.000


descriptive statistics for fields that can be read as numbers


,tenure,MonthlyCharges,TotalCharges,SeniorCitizen
count,7043.000,7043.000,7032.000,7043.000
mean,32.371,64.762,2283.300,0.162
std,24.559,30.090,2266.771,0.369
min,0.000,18.250,18.800,0.000
25%,9.000,35.500,401.450,0.000
50%,29.000,70.350,1397.475,0.000
75%,55.000,89.850,3794.738,0.000
max,72.000,118.750,8684.800,1.000


## 2. Exploratory analysis

Counts below use the raw file. Plots of charges use a numeric conversion that does not yet fill missing values, so the blanks are not silently turned into zeros before we look at them.

Correlation is computed only among the numeric fields. A correlation is not a cause, and it ignores the categorical service fields.

In [ ]:
target = raw[TARGET_COL].astype(str).str.strip()
print("class balance")
display(pd.DataFrame({
    "count": target.value_counts(),
    "percent": (target.value_counts(normalize=True) * 100).round(2),
}))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, column in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    numeric_as_numbers[column].dropna().plot(kind="hist", bins=30, ax=axis, title=column)
fig.tight_layout()
plt.show()

class balance


,count,percent
Churn,,
No,5174,73.46
Yes,1869,26.54


C:\Users\Ayush\AppData\Local\Temp\ipykernel_20384\105812996.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
plot_frame = numeric_as_numbers.copy()
plot_frame[TARGET_COL] = target
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, column in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    plot_frame.boxplot(column=column, by=TARGET_COL, ax=axis)
    axis.set_title(column)
    axis.set_xlabel(TARGET_COL)
fig.suptitle("")
fig.tight_layout()
plt.show()

C:\Users\Ayush\AppData\Local\Temp\ipykernel_20384\752868887.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def churn_rate_table(frame, column):
    grouped = frame.groupby(column, dropna=False)[TARGET_COL]
    table = pd.DataFrame({
        "customers": grouped.size(),
        "churn_yes": grouped.apply(lambda values: values.astype(str).str.strip().eq("Yes").sum()),
    })
    table["churn_rate_percent"] = (table["churn_yes"] / table["customers"] * 100).round(2)
    return table.sort_values("churn_rate_percent", ascending=False)

eda = raw.copy()
eda[TARGET_COL] = target
eda["tenure_number"] = numeric_as_numbers["tenure"]
eda["tenure_band"] = pd.cut(
    eda["tenure_number"],
    bins=[-0.01, 0, 12, 24, 48, 100],
    labels=["0", "1-12", "13-24", "25-48", "49+"],
)
for column in ["Contract", "tenure_band", "PaymentMethod", "InternetService", "TechSupport"]:
    print(column)
    display(churn_rate_table(eda, column))

Contract


,customers,churn_yes,churn_rate_percent
Contract,,,
Month-to-month,3875,1655,42.71
One year,1473,166,11.27
Two year,1695,48,2.83


tenure_band


,customers,churn_yes,churn_rate_percent
tenure_band,,,
1-12,2175,1037,47.68
13-24,1024,294,28.71
25-48,1594,325,20.39
49+,2239,213,9.51
0,11,0,0.00


PaymentMethod


,customers,churn_yes,churn_rate_percent
PaymentMethod,,,
Electronic check,2365,1071,45.29
Mailed check,1612,308,19.11
Bank transfer (automatic),1544,258,16.71
Credit card (automatic),1522,232,15.24


InternetService


,customers,churn_yes,churn_rate_percent
InternetService,,,
Fiber optic,3096,1297,41.89
DSL,2421,459,18.96
No,1526,113,7.40


TechSupport


,customers,churn_yes,churn_rate_percent
TechSupport,,,
No,3473,1446,41.64
Yes,2044,310,15.17
No internet service,1526,113,7.40


In [8]:
charges = numeric_as_numbers[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
charges[TARGET_COL] = target
sample = charges.dropna().sample(n=min(1500, charges.dropna().shape[0]), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for label, marker in [("No", "."), ("Yes", "x")]:
    part = sample[sample[TARGET_COL] == label]
    axes[0].scatter(part["tenure"], part["MonthlyCharges"], s=8, alpha=0.4, label=label, marker=marker)
    axes[1].scatter(part["tenure"], part["TotalCharges"], s=8, alpha=0.4, label=label, marker=marker)
axes[0].set_xlabel("tenure")
axes[0].set_ylabel("MonthlyCharges")
axes[1].set_xlabel("tenure")
axes[1].set_ylabel("TotalCharges")
axes[0].legend()
axes[1].legend()
fig.tight_layout()
plt.show()

print("Pearson correlation among numeric fields. This does not include contract or service category.")
display(charges[["tenure", "MonthlyCharges", "TotalCharges"]].corr(numeric_only=True).round(3))

Pearson correlation among numeric fields. This does not include contract or service category.


C:\Users\Ayush\AppData\Local\Temp\ipykernel_20384\885873464.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,tenure,MonthlyCharges,TotalCharges
tenure,1.000,0.248,0.826
MonthlyCharges,0.248,1.000,0.651
TotalCharges,0.826,0.651,1.000


In [9]:
print("Contract by InternetService, with churn rate")
cross = eda.copy()
cross["churn_yes"] = cross[TARGET_COL].eq("Yes")
pivot = cross.pivot_table(index="Contract", columns="InternetService", values="churn_yes", aggfunc="mean")
display((pivot * 100).round(2))

Contract by InternetService, with churn rate


InternetService,DSL,Fiber optic,No
Contract,,,
Month-to-month,32.22,54.61,18.89
One year,9.30,19.29,2.47
Two year,1.91,7.23,0.78


### How to read the exploratory plots

The printed tables are the evidence. In particular, compare the churn rate for month-to-month contracts with the longer contracts, and compare fiber with the other internet values. A higher rate in one group is a reason to keep that column in the model. It is not a reason to say that the column causes churn.

`TotalCharges` will usually track tenure, because it accumulates over time. A strong correlation between those two columns means they carry overlapping information. Both are still kept in the baseline, because they are the original billing fields and the experiment is not a feature-deletion study.

## 3. Missing values and data quality

`TotalCharges` is stored as text in the raw file. The standard question is whether the non-numeric entries are new accounts. The next cell counts them and prints the matching tenure values before any fill.

In [10]:
total_numeric = pd.to_numeric(raw["TotalCharges"], errors="coerce")
tenure_numeric = pd.to_numeric(raw["tenure"], errors="coerce")
blank = total_numeric.isna()
print("non-numeric TotalCharges", int(blank.sum()))
print("tenure value counts among those rows")
display(tenure_numeric[blank].value_counts(dropna=False).to_frame("rows"))
display(raw.loc[blank, [ID_COL, "tenure", "MonthlyCharges", "TotalCharges", "Contract", TARGET_COL]])

cleaned = clean_features(raw)
print("missing TotalCharges after the tenure-0 rule", int(cleaned["TotalCharges"].isna().sum()))
print("rows removed", 0)
print("customerID still present", ID_COL in cleaned.columns)
print("customerID used as a predictor", ID_COL in FEATURE_COLUMNS)

non-numeric TotalCharges 11
tenure value counts among those rows


,rows
tenure,
0,11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20,,Two year,No
5218,2923-ARZLG,0,19.7,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


missing TotalCharges after the tenure-0 rule 0
rows removed 0
customerID still present True
customerID used as a predictor False


If every non-numeric `TotalCharges` row has tenure 0, replacing those blanks with 0 matches the meaning of an accumulated charge: there has been no bill yet. Mean or median imputation would invent a bill for a brand-new account. The cleaning function applies 0 only in that tenure-0 case. Any other missing numeric value would stay missing and be imputed inside a training fold. This notebook does not drop the affected rows.

## 4. Outlier analysis

The interquartile rule flags points outside the usual whiskers. A flag is a candidate, not an error. Tenure is bounded by the length of the customer relationship, and a high monthly charge can be a real plan.

In [11]:
def iqr_flags(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    spread = q3 - q1
    low = q1 - 1.5 * spread
    high = q3 + 1.5 * spread
    flagged = values[(values < low) | (values > high)]
    return {"q1": float(q1), "q3": float(q3), "low": float(low), "high": float(high), "flagged_count": int(flagged.shape[0])}

for column in ["tenure", "MonthlyCharges", "TotalCharges"]:
    print(column, iqr_flags(numeric_as_numbers[column]))

print("No rows are removed. The training pipeline does not clip these fields.")
print("tenure min/max", float(tenure_numeric.min()), float(tenure_numeric.max()))
print("MonthlyCharges min/max", float(pd.to_numeric(raw["MonthlyCharges"], errors="coerce").min()), float(pd.to_numeric(raw["MonthlyCharges"], errors="coerce").max()))

tenure {'q1': 9.0, 'q3': 55.0, 'low': -60.0, 'high': 124.0, 'flagged_count': 0}
MonthlyCharges {'q1': 35.5, 'q3': 89.85, 'low': -46.02499999999999, 'high': 171.375, 'flagged_count': 0}
TotalCharges {'q1': 401.45, 'q3': 3794.7375, 'low': -4688.481250000001, 'high': 8884.66875, 'flagged_count': 0}
No rows are removed. The training pipeline does not clip these fields.
tenure min/max 0.0 72.0
MonthlyCharges min/max 18.25 118.75


## 5. Leakage-safe preprocessing

The development/test split is stratified and uses a fixed seed. The test frame is not passed into the comparison routine until `run_experiments` has finished selection. Inside that routine, each fold fits its own imputer, scaler, and encoder.

The cell below shows the idea on a toy column: the median used to fill a gap is the training median, not the median of the combined development and test rows.

In [12]:
model_frame = cleaned.copy()
y = encode_target(model_frame)
X = model_frame[FEATURE_COLUMNS]
x_dev, x_test, y_dev, y_test = split_development_test(X, y)
print("development", len(x_dev), "test", len(x_test))
print("development churn rate", round(float(y_dev.mean()), 4), "test churn rate", round(float(y_test.mean()), 4))
assert set(x_dev.index).isdisjoint(x_test.index)

toy = pd.DataFrame({"tenure": [1, 2, 3, None], "holdout": [10, 10, 10, 10]})
train_median = toy.loc[0:2, "tenure"].median()
print("median fitted on the first three rows", float(train_median))
print("median if the holdout value were included", float(toy["tenure"].median()))
print("preprocessor for the baseline configuration")
print(build_preprocessor("baseline"))

development 5634 test 1409
development churn rate 0.2654 test churn rate 0.2654
median fitted on the first three rows 2.0
median if the holdout value were included 2.0
preprocessor for the baseline configuration
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'SeniorCitizen']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                               

## 6. Feature engineering

Two setups are compared later with the same folds.

- Baseline: the original predictors after the cleaning above.
- Engineered: those predictors plus `tenure_group` and `optional_service_count`.

`tenure_group` uses fixed edges at 0, 12, 24, and 48 months. Twelve and 24 match the one-year and two-year contract lengths in this file. The edges are not quantiles, so they do not learn from the target.

`optional_service_count` counts `Yes` among OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, and StreamingMovies. `No` and `No internet service` count as zero. Phone-line fields are not included.

In [13]:
engineered = add_domain_features(model_frame)
print("optional-service columns", OPTIONAL_SERVICE_COLUMNS)
print("count distribution")
display(engineered["optional_service_count"].value_counts().sort_index().to_frame("customers"))
print("churn rate by tenure group")
engineered[TARGET_COL] = model_frame[TARGET_COL]
display(churn_rate_table(engineered, "tenure_group"))
print("churn rate by optional service count")
display(churn_rate_table(engineered, "optional_service_count"))
print("customerID in engineered predictors", ID_COL in FEATURE_COLUMNS)

optional-service columns ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
count distribution


,customers
optional_service_count,
0,2219
1,966
2,1033
3,1118
4,852
5,571
6,284


churn rate by tenure group


,customers,churn_yes,churn_rate_percent
tenure_group,,,
1-12 months,2175,1037,47.68
13-24 months,1024,294,28.71
25-48 months,1594,325,20.39
49+ months,2239,213,9.51
0 months,11,0,0.00


churn rate by optional service count


,customers,churn_yes,churn_rate_percent
optional_service_count,,,
1,966,442,45.76
2,1033,370,35.82
3,1118,306,27.37
4,852,190,22.30
0,2219,475,21.41
5,571,71,12.43
6,284,15,5.28


customerID in engineered predictors False


## 7 to 11. Comparison, threshold, and final test

The next cell runs the shared experiment. It can take several minutes.

What it does, in order:

1. Stratified 5-fold cross-validation on the development set for each feature setup, model, and imbalance setting that imports.
2. Selection by mean average precision. Precision, recall, and F1 at 0.50 are recorded and are not the selection rule.
3. A separate single-split ROC-AUC ranking on the development set only. That ranking is a diagnostic. It does not choose the saved model.
4. Out-of-fold probabilities for the selected pipeline. The threshold is the one with the highest F1 on those probabilities.
5. One refit on the full development set, then one evaluation on the untouched test set.

Class weighting is the main imbalance setting. Unweighted logistic regression, unweighted LightGBM, and SMOTE inside the training folds of a logistic pipeline are included when the libraries import. SMOTE is not applied to validation or test rows. The three boosters are also combined in a soft vote when all three import. The vote is kept only if its mean average precision is the best row; it is not assumed to win.

In [14]:
report = run_experiments(csv_path=DATA, models_dir=ROOT / "models", save=True)
cv = pd.DataFrame(report["cv_results"])
ranked = cv.dropna(subset=["average_precision_mean"]).sort_values(
    ["average_precision_mean", "average_precision_std"],
    ascending=[False, True],
)
columns = [
    "feature_config",
    "model",
    "imbalance",
    "average_precision_mean",
    "average_precision_std",
    "roc_auc_mean",
    "roc_auc_std",
    "precision_at_0_50_mean",
    "recall_at_0_50_mean",
    "f1_at_0_50_mean",
    "brier_mean",
]
display(ranked[columns].round(4))

Cross-validating baseline | logistic_regression | class_weight


  AP 0.6600 ± 0.0218 ROC-AUC 0.8460
Cross-validating baseline | random_forest | class_weight


  AP 0.6553 ± 0.0210 ROC-AUC 0.8448
Cross-validating baseline | xgboost | class_weight


  AP 0.6688 ± 0.0252 ROC-AUC 0.8481
Cross-validating baseline | lightgbm | class_weight


  AP 0.6667 ± 0.0301 ROC-AUC 0.8471
Cross-validating baseline | catboost | class_weight


...

  AP 0.6624 ± 0.0165 ROC-AUC 0.8451
Cross-validating engineered | soft_vote_xgb_cat_lgbm | class_weight


  AP 0.6704 ± 0.0233 ROC-AUC 0.8494
Ranking a single development split by ROC-AUC (diagnostic only).


Fitting the selected pipeline on the full development set.


Saved engineered / soft_vote_xgb_cat_lgbm / class_weight with threshold 0.55


,feature_config,model,imbalance,average_precision_mean,average_precision_std,roc_auc_mean,roc_auc_std,precision_at_0_50_mean,recall_at_0_50_mean,f1_at_0_50_mean,brier_mean
17,engineered,soft_vote_xgb_cat_lgbm,class_weight,0.6704,0.0233,0.8494,0.0123,0.5263,0.8000,0.6348,0.1590
8,baseline,soft_vote_xgb_cat_lgbm,class_weight,0.6703,0.0245,0.8495,0.0131,0.5272,0.7940,0.6335,0.1588
6,baseline,lightgbm,none,0.6696,0.0237,0.8480,0.0124,0.6632,0.5237,0.5850,0.1338
2,baseline,xgboost,class_weight,0.6688,0.0252,0.8481,0.0129,0.5258,0.7893,0.6311,0.1589
11,engineered,xgboost,class_weight,0.6684,0.0236,0.8471,0.0119,0.5298,0.7920,0.6348,0.1593
13,engineered,catboost,class_weight,0.6678,0.0211,0.8495,0.0129,0.5188,0.8047,0.6307,0.1613
12,engineered,lightgbm,class_weight,0.6676,0.0255,0.8472,0.0125,0.5234,0.7946,0.6311,0.1596
4,baseline,catboost,class_weight,0.6673,0.0211,0.8487,0.0128,0.5243,0.8040,0.6345,0.1611
3,baseline,lightgbm,class_weight,0.6667,0.0301,0.8471,0.0134,0.5249,0.7913,0.6311,0.1596
14,engineered,logistic_regression,none,0.6651,0.0149,0.8465,0.0130,0.6679,0.5358,0.5939,0.1346


In [15]:
comparable = ranked[ranked["imbalance"] == "class_weight"]
baseline_best = comparable[comparable["feature_config"] == "baseline"].iloc[0]
engineered_best = comparable[comparable["feature_config"] == "engineered"].iloc[0]
print("Best class-weighted baseline", baseline_best["model"], round(baseline_best["average_precision_mean"], 4))
print("Best class-weighted engineered", engineered_best["model"], round(engineered_best["average_precision_mean"], 4))
print(
    "Engineered minus baseline on those two rows",
    round(engineered_best["average_precision_mean"] - baseline_best["average_precision_mean"], 4),
)
same_model = comparable.pivot_table(index="model", columns="feature_config", values="average_precision_mean")
if {"baseline", "engineered"}.issubset(same_model.columns):
    same_model["engineered_minus_baseline"] = same_model["engineered"] - same_model["baseline"]
    display(same_model.round(4))
print("Selected by cross-validation", report["selected_feature_config"], report["selected_model"], report["selected_imbalance"])
print("Single development split would have ranked first by ROC-AUC:")
display(pd.DataFrame(report["single_development_split_roc_auc_ranking"]).head(3).round(4))

Best class-weighted baseline soft_vote_xgb_cat_lgbm 0.6703
Best class-weighted engineered soft_vote_xgb_cat_lgbm 0.6704
Engineered minus baseline on those two rows 0.0001


feature_config,baseline,engineered,engineered_minus_baseline
model,,,
catboost,0.6673,0.6678,0.0005
lightgbm,0.6667,0.6676,0.0008
logistic_regression,0.6600,0.6641,0.0041
random_forest,0.6553,0.6569,0.0016
soft_vote_xgb_cat_lgbm,0.6703,0.6704,0.0001
xgboost,0.6688,0.6684,-0.0004


Selected by cross-validation engineered soft_vote_xgb_cat_lgbm class_weight
Single development split would have ranked first by ROC-AUC:


,feature_config,model,imbalance,holdout_roc_auc,holdout_average_precision
0,engineered,catboost,class_weight,0.8531,0.6800
1,baseline,soft_vote_xgb_cat_lgbm,class_weight,0.8523,0.6731
2,engineered,soft_vote_xgb_cat_lgbm,class_weight,0.8517,0.6740


## 10. Thresholds

The curve uses out-of-fold development probabilities for the selected pipeline. The marked threshold is the maximum F1 on that curve. It was not chosen by looking at the test set.

A different business cost would choose a different point. Maximising F1 is a documented policy for this prototype, not a universal operating point.

In [16]:
curve = pd.DataFrame(report["threshold_curve"])
fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(curve["threshold"], curve["precision"], label="precision")
axis.plot(curve["threshold"], curve["recall"], label="recall")
axis.plot(curve["threshold"], curve["f1"], label="F1")
chosen = report["decision_threshold"]
axis.axvline(chosen, color="black", linestyle="--", label=f"selected {chosen:.2f}")
axis.set_xlabel("threshold")
axis.set_ylabel("score")
axis.legend()
fig.tight_layout()
plt.show()
display(pd.DataFrame([report["threshold_selection"]]).round(4))
print("out-of-fold metrics at the selected threshold")
display(pd.DataFrame([report["oof_metrics_at_selected_threshold"]]).T)

C:\Users\Ayush\AppData\Local\Temp\ipykernel_20384\1237892212.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,threshold,precision,recall,f1,flagged,flagged_rate,false_positives,false_negatives
0,0.55,0.5582,0.7572,0.6426,2028,0.36,896,363


out-of-fold metrics at the selected threshold


,0
threshold,0.55
average_precision,0.667166
roc_auc,0.848969
precision,0.558185
recall,0.757191
f1,0.642634
balanced_accuracy,0.770357
brier,0.159016
confusion_matrix,"[[3243, 896], [363, 1132]]"
flagged,2028


## 11. Final test evaluation

These numbers are the one test-set look. They are not compared in order to change the selected model.

In [17]:
print("final test metrics")
display(pd.DataFrame([report["final_test_metrics"]]).T)
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

model_frame = load_clean_frame(DATA)
y = encode_target(model_frame)
X = model_frame[FEATURE_COLUMNS]
_, x_test, _, y_test = split_development_test(X, y)
import joblib
pipeline = joblib.load(ROOT / "models" / "churn_pipeline.joblib")
test_proba = pipeline.predict_proba(x_test)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[0], name="selected pipeline")
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=axes[1], name="selected pipeline")
fig.tight_layout()
plt.show()

final test metrics


,0
threshold,0.55
average_precision,0.667121
roc_auc,0.84655
precision,0.541109
recall,0.756684
f1,0.630992
balanced_accuracy,0.7624
brier,0.161153
confusion_matrix,"[[795, 240], [91, 283]]"
flagged,523


C:\Users\Ayush\AppData\Local\Temp\ipykernel_20384\522019107.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Explainability

SHAP is computed for individual development-set customers through the saved pipeline. A positive value pushes the score toward churn. It is not a causal effect, and it is not a reason to promise that a retention offer will work.

If the explainer cannot run for the saved model, the cell prints the error and does not invent factors.

In [18]:
examples = x_dev.head(3)
for _, row in examples.iterrows():
    payload = row.to_dict()
    payload[ID_COL] = "notebook-example"
    explanation = explain_one(payload)
    print(explanation.get("method") or explanation.get("error"))
    if explanation.get("ok"):
        display(pd.DataFrame(explanation["factors"]))
    else:
        print(explanation)

mean of shap.TreeExplainer values across soft-vote members; this is not a single joint SHAP decomposition of the averaged probability


C:\Users\Ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,label,shap_value,effect
0,cat__Contract_Month-to-month,Contract Month-to-month,0.4997,increases predicted churn risk
1,num__tenure,tenure,-0.3490,decreases predicted churn risk
2,cat__PaymentMethod_Electronic check,PaymentMethod Electronic check,0.3185,increases predicted churn risk
3,cat__InternetService_Fiber optic,InternetService Fiber optic,-0.2830,decreases predicted churn risk
4,num__TotalCharges,TotalCharges,-0.2010,decreases predicted churn risk


mean of shap.TreeExplainer values across soft-vote members; this is not a single joint SHAP decomposition of the averaged probability


C:\Users\Ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,label,shap_value,effect
0,cat__Contract_Month-to-month,Contract Month-to-month,0.6672,increases predicted churn risk
1,cat__InternetService_Fiber optic,InternetService Fiber optic,0.2746,increases predicted churn risk
2,cat__OnlineSecurity_No,OnlineSecurity No,-0.2617,decreases predicted churn risk
3,cat__PaymentMethod_Electronic check,PaymentMethod Electronic check,-0.1994,decreases predicted churn risk
4,cat__PaperlessBilling_No,PaperlessBilling No,-0.1560,decreases predicted churn risk


mean of shap.TreeExplainer values across soft-vote members; this is not a single joint SHAP decomposition of the averaged probability


C:\Users\Ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,label,shap_value,effect
0,cat__Contract_Month-to-month,Contract Month-to-month,-0.9719,decreases predicted churn risk
1,cat__Contract_Two year,Contract Two year,-0.4476,decreases predicted churn risk
2,cat__OnlineSecurity_No,OnlineSecurity No,-0.2853,decreases predicted churn risk
3,cat__InternetService_Fiber optic,InternetService Fiber optic,-0.1912,decreases predicted churn risk
4,num__MonthlyCharges,MonthlyCharges,-0.1909,decreases predicted churn risk


## 13. What was saved

`run_experiments` already wrote:

- `models/churn_pipeline.joblib` — the development-set fit of the selected pipeline
- `models/shap_background.npy` — a development-set background sample for linear SHAP
- `models/training_metrics.json` — cross-validation, the threshold, the test metrics, and the feature definitions

Inference loads that file. It does not fit a new model per request. The raw API fields stay the original customer columns; the pipeline adds engineered fields only when the selected configuration needs them.

In [19]:
saved = ROOT / "models" / "training_metrics.json"
print(saved.exists(), saved.stat().st_size if saved.exists() else 0)
print("decision threshold", report["decision_threshold"])
print("selection rule")
print(report["selection_rule"])

True 46455
decision threshold 0.55
selection rule
Highest mean stratified 5-fold average precision on the development set. The 20% test set was not used for feature, model, or threshold selection.
